In [17]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chat_models.base import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import pandas as pd

load_dotenv()

True

In [18]:
# OpenAI:
# llm = init_chat_model("openai:gpt-3.5-turbo")

# embedding_model = "text-embedding-3-small" # outputs 1536 dimensions by default
# embedding = OpenAIEmbeddings(model=embedding_model)

# Ollama:
llm = ChatOllama(model="gemma3:12b")
embedding = OllamaEmbeddings(model="qwen3-embedding:8b")

In [19]:

fnx_doc = []

try:
    policies_table = pd.read_excel("../DataIngestParsing/data/csv_excel/policies_table.xlsx")
    users_table = pd.read_excel("../DataIngestParsing/data/csv_excel/users_table.xlsx")
    cars_tabel = pd.read_excel("../DataIngestParsing/data/csv_excel/cars_table.xlsx")

    for _, row in policies_table.iterrows():
        user_matched = users_table[users_table["Id"] == row["CustomerId"]]
        user_row = user_matched.iloc[0] if not user_matched.empty else pd.Series(dtype=object)

        car_matched = cars_tabel[cars_tabel["PolicyId"] == row["Id"]]
        car_row = car_matched.iloc[0] if not car_matched.empty else pd.Series(dtype=object)

        text = f"""
        פוליסה מספר {row.get('PolicyNumber', '')},
        שנת חיתום: {row.get("ShnatChitum", '')}
        ענף: {row.get("Anaf", '')}
        תאריך תחילת הפוליסה: {row.get("PolicyStartDate", '')}
        תאריך סיום הפוליסה: {row.get("PolicyEndDate", '')}
        מזהה לקוח: {row.get('CustomerId', '')}

        בעל הפוליסה: {user_row.get('FirstName', '')} {user_row.get('LastName', '')}
        תעודת זהות: {user_row.get('GovId', '')}

        כתובת בעל הפוליסה:
        {user_row.get('CityDesc', '')}, {user_row.get('StreetDesc', '')} {user_row.get('HouseNumber', '')}

        תאריך לידה: {user_row.get('BirthdayDate', '')}
        שנת הוצאת רישיון: {user_row.get('YearlicenseIssued', '')}
        כתובת אימייל: {user_row.get('Email', '')}

        הפוליסה מנוהלת על ידי הסוכן: {row.get('AgentName', '')}
        מספר סוכן: {row.get('AgentId', '')}
        הפוליסה נוצרה על ידי: {row.get('CreatedBy', '')}

        סטטוס הפוליסה: {"לא פעילה" if row.get("IsActive") == 0 else "פעילה"}.
        מספר התביעות שהיו לפוליסה: {row.get("NumberOfClaims", '')}
        מספר השלילות שהיו לפוליסה: {row.get("NumberOfDenials", '')}

         לפוליסה זו משויך רכב:
         {car_row.get("ModelName", '')} {car_row.get("ManufactureYear", '')} מספר {car_row.get("CarNumber", '')} - בעל הרכב {car_row.get("CarOwnerDetails", '')} - תעריף לקילומטר {car_row.get("KmPremium", '')} ש"ח - הרכב מבוטח בביטוח {car_row.get("InsuranceDesc", '')} - שווי רכב משוער {car_row.get("CarPrice", '')} ש"ח
        """

        # Build metadata
        metadata = {
             "מספר פוליסה מלא": row.get('PolicyId', ''),
             "מספר הפוליסה": row.get('PolicyNumber', ''),
        }

        fnx_doc.append(
            Document(
                page_content=text,
                metadata=metadata
            )
        )

    print(f"The embedding data will look like that: {fnx_doc[52]}")
except Exception as e:
    print(f"Error loading PyPDF: {e}")


The embedding data will look like that: page_content='
        פוליסה מספר 1006510,
        שנת חיתום: 25
        ענף: 112
        תאריך תחילת הפוליסה: 2025-09-03 00:00:00
        תאריך סיום הפוליסה: 2026-05-06 10:49:25.043000
        מזהה לקוח: 37655

        בעל הפוליסה: רום בן הרוש
        תעודת זהות: 323826461

        כתובת בעל הפוליסה:
        הוד השרון, דוד אלעזר 62

        תאריך לידה: 2004-02-20 00:00:00
        שנת הוצאת רישיון: 2021
        כתובת אימייל: izometria@bezeqint.net

        הפוליסה מנוהלת על ידי הסוכן: לוי עזר גיא
        מספר סוכן: 34520
        הפוליסה נוצרה על ידי: *אלינה טורצקי

        סטטוס הפוליסה: פעילה.
        מספר התביעות שהיו לפוליסה: 0.0
        מספר השלילות שהיו לפוליסה: 0.0

         לפוליסה זו משויך רכב:
         טויוטה ראב 4 (2500) E-MOTION פלאג אין 6 2022 מספר 86934002 - בעל הרכב אלי בן הרוש - תעריף לקילומטר 0.71 ש"ח - הרכב מבוטח בביטוח מקיף - שווי רכב משוער 197999.0 ש"ח
        ' metadata={'מספר פוליסה מלא': 251001121006510, 'מספר הפוליסה': 100

In [20]:
 # Vector store: ChromaDB (Chroma.from_documents) automatically infers the dimension from the first embedding it receives — you don't need to configure it manually. It adapts to whatever model you pass.

persist_directory = "./vectorstore"
os.makedirs(persist_directory, exist_ok=True)
vector_store = Chroma.from_documents(
    documents=fnx_doc,
    embedding=embedding,
    persist_directory=persist_directory, # This will create vectorstore for the data
    collection_name="core"
)

print(f"Created vector store with {len(vector_store)} vectors")

Created vector store with 2346 vectors


In [21]:
# Similarity TEST: Similarity search works on page_content, NOT metadata
query = "ישי אמסלם"

# similar_docs = vector_store.similarity_search(query, k=2)
similar_docs_with_scores = vector_store.similarity_search_with_score(query, k=2)

# scores with ChromaDB:
# Closer to 0 = Similar result
# Zero means identical vectors

# print(similar_docs[0])
# print("---------")
print(similar_docs_with_scores)

[(Document(metadata={'מספר הפוליסה': 1041630, 'מספר פוליסה מלא': 260741121041630}, page_content='\n        פוליסה מספר 1041630,\n        שנת חיתום: 26\n        ענף: 112\n        תאריך תחילת הפוליסה: 2026-05-02 00:00:00\n        תאריך סיום הפוליסה: 2027-04-30 23:59:59\n        מזהה לקוח: 70587\n\n        בעל הפוליסה: אמיר מדמוני\n        תעודת זהות: 331830836\n\n        כתובת בעל הפוליסה:\n        נטור, הכרם 56\n\n        תאריך לידה: 2009-04-26 00:00:00\n        שנת הוצאת רישיון: 2026\n        כתובת אימייל: amirmadmoni@gmail.com\n\n        הפוליסה מנוהלת על ידי הסוכן: נח סוכנות ביטוח בע"מ\n        מספר סוכן: 33258\n        הפוליסה נוצרה על ידי: אמיר מדמוני\n\n        סטטוס הפוליסה: פעילה.\n        מספר התביעות שהיו לפוליסה: nan\n        מספר השלילות שהיו לפוליסה: nan\n\n         לפוליסה זו משויך רכב:\n         טויוטה היילקס אדוונצ\'ר דיזל 2910 ק"ג55 2022 מספר 14879103 - בעל הרכב יעקב מדמוני אביחי - תעריף לקילומטר 0.81 ש"ח - הרכב מבוטח בביטוח מקיף - שווי רכב משוער 169001.0 ש"ח\n        '

In [22]:
custom_prompt = """
        You are a professional customer service representative for Phoenix Insurance Company.
        ***Always respond in Hebrew regardless of the language used in the question***.

        Your goal is to provide accurate information about the company's policies.

        Guidelines:
        * Use a professional, patient, and respectful tone.
        * If asked analytical questions, respond that you are not an analytics tool — DO NOT RETURN ANY DATA!
        * If information is missing or not found in the system — state it clearly.
        * Do not fabricate information that does not exist in the provided data.
        * If multiple similar results are found — ask for additional details to identify the correct one.
        * If the question is unrelated to Phoenix or the data in the system — politely respond that you cannot assist with that topic.
        * Phrase answers in a natural, human way — not as a raw technical data list.

        Example response style:
        * "Your policy is active until 11/05/2026..."
        * "The agent handling your policy is SMART הפניקס..."
        * "No policy was found matching the provided details..."

        ***Always respond in Hebrew***.

        Context: {context}
    """

In [23]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
import re
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import Field
from typing import Any

# BM25 retriever (Exact mach for names - names are not embed great so the semantic search is weak)
bm25_retriever = BM25Retriever.from_documents(fnx_doc, k=3)
# Dense retriever - Regular similarity search
semantic_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 80/50 blend — tune weights based on your results
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.8, 0.2]
)

class HybridRetriever(BaseRetriever):
    vector_store: Any = Field(...)
    ensemble: Any = Field(...)
    k: int = Field(default=3)

    def _get_relevant_documents(self, query: str) -> list[Document]:
        # 1. Exact policy number lookup via metadata
        policy_match = re.search(r'\b(\d{6,10})\b', query)
        if policy_match:
            policy_number = int(policy_match.group(1))
            results = self.vector_store.get(where={"הסילופה רפסמ": policy_number})
            docs = [
                Document(page_content=pc, metadata=meta)
                for pc, meta in zip(results["documents"], results["metadatas"])
            ]
            if docs:
                return docs

        # 2. BM25 + semantic ensemble for everything else (names, questions, etc.)
        return self.ensemble.invoke(query)

retriever = HybridRetriever(vector_store=vector_store, ensemble=ensemble_retriever)

In [24]:
prompt = ChatPromptTemplate.from_messages([
    ("system", custom_prompt),
    ("human", "{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n        You are a professional customer service representative for Phoenix Insurance Company.\n        ***Always respond in Hebrew regardless of the language used in the question***.\n\n        Your goal is to provide accurate information about the company\'s policies.\n\n        Guidelines:\n        * Use a professional, patient, and respectful tone.\n        * If asked analytical questions, respond that you are not an analytics tool — DO NOT RETURN ANY DATA!\n        * If information is missing or not found in the system — state it clearly.\n        * Do not fabricate information that does not exist in the provided data.\n        * If multiple similar results are found — ask for additional details to identify the correct one.\n        * If the question i

In [25]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)


In [26]:
response = rag_chain.invoke({ "input": "מה הכתובת של בעל הפוליסה 1029855?" })
print(response.get("answer"))

כתובת בעל הפוליסה 1029855, זוהר ישורון, היא פרדס חנה-כרכור, היובל 14.


In [27]:
response = rag_chain.invoke({ "input": "האם קיים לקוח בשם יונתן מורל?" })
print(response.get("answer"))

כן, קיים לקוח בשם יונתן מורל. פוליסה מספר 1005669 שייכת ליונתן מורל, תעודת זהות 325910917, המתגורר באשקלון, חזון עובדיה 31. הפוליסה פעילה ותוקפה עד ה-12 במאי 2026.


In [28]:
response = rag_chain.invoke({ "input": "האם לאגם וקנין יש פוליסה פעילה?" })
print(response.get("answer"))

כן, לאגם וקנין יש שתי פוליסות פעילות.

*   **פוליסה מספר 1002842:** הפוליסה פעילה מתאריך 2026-06-01 ועד 2027-05-31. הרכב המשויך לפוליסה הוא סיטרואן C-1 FEEL.
*   **פוליסה מספר 1008091:** הפוליסה פעילה מתאריך 2026-06-01 ועד 2027-05-31. הרכב המשויך לפוליסה הוא פיג'ו 3008 אקטיב.

האם תרצה מידע נוסף על אחת הפוליסות?


In [29]:
response = rag_chain.invoke({ "input": "מי בעל הפוליסה 1015082?" })
print(response.get("answer"))

בעל הפוליסה מספר 1015082 הוא עילי מזרחי.


In [30]:
response = rag_chain.invoke({ "input": "מה המייל של ינון אליה דוידי?" })
print(response.get("answer"))

כתובת האימייל של ינון אליה דוידי היא: inon211280@gmail.com.
